# Inserción continua de datos — MySQL (`eventos_juegos`)

Este notebook simula un flujo **near real-time**: cada pocos segundos inserta una fila nueva en la tabla `eventos_juegos` de la base `staging_steam` (MySQL). Logstash, corriendo en paralelo con `mysql_logstash.conf`, va a detectar estas filas nuevas en su siguiente ciclo (`schedule` cada 10 segundos) y las va a indexar automáticamente en Elasticsearch.

**Requisitos:**
```
pip install mysql-connector-python Faker
```

In [1]:
import mysql.connector
import random
import time
from datetime import datetime
from faker import Faker

fake = Faker()

In [2]:
DB_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "user": "root",
    "password": "miclave",
    "database": "staging_steam",
}

## Datos de referencia para simular eventos realistas

`appid` y `evento` se mantienen como dominios controlados (deben ser coherentes con el proyecto). `pais` e `ip_origen` se generan con Faker en la función `generar_evento()`.

La función está escrita como código puro (sin dependencias del entorno del notebook) para poder reutilizarla más adelante en un DAG de Airflow sin modificarla.

In [3]:
APPIDS_MUESTRA = [730, 578080, 570, 271590, 440, 105600, 252490, 4000, 1091500, 292030]
TIPOS_EVENTO = ["compra", "review", "jugada", "descarga", "actualizacion"]


def generar_evento():
    return {
        "appid": random.choice(APPIDS_MUESTRA),
        "evento": random.choice(TIPOS_EVENTO),
        "cantidad": random.randint(1, 500),
        "fecha_evento": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "pais": fake.country(),
        "ip_origen": fake.ipv4(),
    }

## Función de inserción

In [4]:
def insertar_evento(cursor, evento):
    query = (
        "INSERT INTO eventos_juegos (appid, evento, cantidad, fecha_evento, pais, ip_origen) "
        "VALUES (%(appid)s, %(evento)s, %(cantidad)s, %(fecha_evento)s, %(pais)s, %(ip_origen)s)"
    )
    cursor.execute(query, evento)

## Bucle de simulación (near real-time)

Inserta una fila nueva cada **3 segundos**. Detén la celda (botón de stop / interrumpir kernel) cuando quieras parar la simulación — no hay un número fijo de iteraciones, corre indefinidamente para simular un flujo continuo.

In [5]:
INTERVALO_SEGUNDOS = 3

conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()

print("Insertando eventos cada", INTERVALO_SEGUNDOS, "segundos. Interrumpe el kernel para detener.\n")

contador = 0
try:
    while True:
        evento = generar_evento()
        insertar_evento(cursor, evento)
        conn.commit()
        contador += 1
        print(f"[{contador}] Insertado: {evento}")
        time.sleep(INTERVALO_SEGUNDOS)
except KeyboardInterrupt:
    print("\nSimulación detenida por el usuario.")
finally:
    cursor.close()
    conn.close()
    print(f"Total de eventos insertados en esta sesión: {contador}")

Insertando eventos cada 3 segundos. Interrumpe el kernel para detener.

[1] Insertado: {'appid': 4000, 'evento': 'compra', 'cantidad': 106, 'fecha_evento': '2026-09-06 00:19:55', 'pais': 'Liechtenstein', 'ip_origen': '170.166.195.20'}
[2] Insertado: {'appid': 578080, 'evento': 'review', 'cantidad': 414, 'fecha_evento': '2026-09-06 00:19:58', 'pais': 'Anguilla', 'ip_origen': '104.175.235.226'}
[3] Insertado: {'appid': 570, 'evento': 'actualizacion', 'cantidad': 170, 'fecha_evento': '2026-09-06 00:20:01', 'pais': 'Fiji', 'ip_origen': '23.57.145.66'}
[4] Insertado: {'appid': 105600, 'evento': 'actualizacion', 'cantidad': 284, 'fecha_evento': '2026-09-06 00:20:04', 'pais': 'Germany', 'ip_origen': '48.114.192.20'}
[5] Insertado: {'appid': 292030, 'evento': 'actualizacion', 'cantidad': 240, 'fecha_evento': '2026-09-06 00:20:07', 'pais': 'Gibraltar', 'ip_origen': '23.36.165.232'}
[6] Insertado: {'appid': 578080, 'evento': 'compra', 'cantidad': 486, 'fecha_evento': '2026-09-06 00:20:10', 'pais

OperationalError: 1053 (08S01): Server shutdown in progress